In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import joblib
import numpy as np
import gradio as gr

rf = joblib.load("/content/drive/MyDrive/visa_rf_model.pkl")
gb = joblib.load("/content/drive/MyDrive/visa_gb_model.pkl")
xgb = joblib.load("/content/drive/MyDrive/visa_xgb_model.pkl")
lgbm = joblib.load("/content/drive/MyDrive/visa_lgbm_model.pkl")

stacked_model = joblib.load("/content/drive/MyDrive/final_stacked_model.pkl")


In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/visa_estimator_feature_engineered.csv")
feature_names = list(df.drop("visa_approval_score", axis=1).columns)
feature_names


['education_level',
 'experience_years',
 'annual_salary_usd',
 'company_size',
 'age',
 'country_score',
 'job_skill_score',
 'previous_visas',
 'application_score',
 'season_index',
 'experience_band',
 'salary_per_experience']

In [4]:
def predict(
    education_level,
    experience_years,
    annual_salary_usd,
    company_size,
    age,
    country_score,
    job_skill_score,
    previous_visas,
    application_score,
    season_index,
    experience_band,
    salary_per_experience
):
    X = np.array([[
        education_level,
        experience_years,
        annual_salary_usd,
        company_size,
        age,
        country_score,
        job_skill_score,
        previous_visas,
        application_score,
        season_index,
        experience_band,
        salary_per_experience
    ]], dtype=float)

    rf_pred = rf.predict(X)[0]
    gb_pred = gb.predict(X)[0]
    xgb_pred = xgb.predict(X)[0]
    lgbm_pred = lgbm.predict(X)[0]

    meta_X = np.array([[rf_pred, gb_pred, xgb_pred, lgbm_pred]])
    final_pred = stacked_model.predict(meta_X)[0]

    # Decision + processing days
    if final_pred >= 80:
        decision = "🟢 High Approval Chance"
        processing_days = "10 – 15 days"
    elif final_pred >= 50:
        decision = "🟡 Moderate Approval Chance"
        processing_days = "20 – 30 days"
    else:
        decision = "🔴 Low Approval Chance"
        processing_days = "35 – 50 days"

    lower = round(final_pred - 0.05, 3)
    upper = round(final_pred + 0.05, 3)

    return (
        f"Visa Approval Score: {final_pred:.2f}\n"
        f"Decision: {decision}\n"
        f"Estimated Processing Time: {processing_days}\n"
        f"Prediction Range: {lower} – {upper}"
    )


In [5]:
X_test = np.array([[4, 5, 70000, 3000, 30, 0.85, 0.75, 0, 1, 0, 1, 0
]])

print("RF:", rf.predict(X_test))
print("GB:", gb.predict(X_test))
print("XGB:", xgb.predict(X_test))
print("LGBM:", lgbm.predict(X_test))


RF: [99.10870298]
GB: [99.37787415]
XGB: [99.216446]
LGBM: [99.05545101]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [6]:
import gradio as gr

with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="indigo",
        secondary_hue="slate",
        neutral_hue="slate"
    )
) as app:

    # ================= HOME TAB =================
    with gr.Tab("🏠 Home"):
        gr.Markdown("""
        # 🛂 Visa Approval Prediction System

        ### Smart Decision Support using Machine Learning
        **Milestone 4 – Web Application Deployment**

        This system helps estimate **visa approval chances** and
        **processing time** using a **stacked ensemble ML model**.
        """)

        gr.Markdown("---")

        gr.Markdown("""
        ## ✨ Key Features
        - 🔗 Stacked Ensemble Learning (RF, GB, XGBoost, LightGBM)
        - 🌐 Interactive Web App using **Gradio**
        - 📊 Approval Categories: Low / Moderate / High
        - ⏱️ Estimated Processing Time
        - ⚡ Real-time Predictions
        - 🧠 Engineered Feature Handling
        """)

        gr.Markdown("""
        ## 🔄 How It Works
        1️⃣ User enters applicant & job details
        2️⃣ Base ML models generate predictions
        3️⃣ Meta-model combines results
        4️⃣ Final decision & processing time shown
        """)

    # ================= PREDICTION TAB =================
    with gr.Tab("📊 Prediction"):
        gr.Markdown("## 📊 Visa Approval Prediction")

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 👤 Applicant Profile")
                education_level = gr.Number(label="Education Level (1–5)")
                age = gr.Number(label="Applicant Age")
                experience_years = gr.Number(label="Years of Experience")
                experience_band = gr.Number(label="Experience Band (0–3)")

            with gr.Column():
                gr.Markdown("### 💼 Job & Employer Details")
                annual_salary_usd = gr.Number(label="Annual Salary (USD)")
                salary_per_experience = gr.Number(label="Salary per Experience")
                company_size = gr.Number(label="Company Size")
                job_skill_score = gr.Number(label="Job Skill Score (0–1)")

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 📄 Application & Risk Factors")
                country_score = gr.Number(label="Country Score (0–1)")
                previous_visas = gr.Number(label="Previous Visa Approvals")
                application_score = gr.Number(label="Application Score (0–1)")
                season_index = gr.Number(label="Season Index (1–3)")

        gr.Markdown("---")

        predict_btn = gr.Button("🔍 Predict Visa Outcome", variant="primary")
        reset_btn = gr.Button("♻️ Reset")

        output = gr.Textbox(
            label="📊 Prediction Result",
            lines=6,
            interactive=False
        )

        predict_btn.click(
            fn=predict,
            inputs=[
                education_level,
                experience_years,
                annual_salary_usd,
                company_size,
                age,
                country_score,
                job_skill_score,
                previous_visas,
                application_score,
                season_index,
                experience_band,
                salary_per_experience
            ],
            outputs=output
        )

        reset_btn.click(fn=lambda: "", inputs=[], outputs=output)

    # ================= ABOUT / FAQ TAB =================
    with gr.Tab("ℹ️ About & FAQ"):
        gr.Markdown("""
        ## ℹ️ About the Project

        This project was developed as part of an academic milestone to demonstrate:
        - End-to-end ML pipeline
        - Model deployment
        - User-friendly web interface

        ### 🧠 Technologies Used
        - Python
        - Scikit-learn
        - XGBoost & LightGBM
        - Gradio

        ---
        ## ❓ Frequently Asked Questions

        **Q: Is this a real visa decision system?**
        A: No. This is a predictive academic model for learning purposes.

        **Q: Are all parameters required in real life?**
        A: Some are user-provided, others are system-derived or engineered.

        **Q: Why use ensemble learning?**
        A: To improve accuracy and stability over single models.
        """)

app.launch(share=True)


/tmp/ipython-input-2176196411.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://087753b880d1268563.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
